# Step 6 — Geographic Map (Folium)
Interactive world map of data centre locations coloured by water stress.

In [1]:
import folium
from folium.plugins import MarkerCluster
import pandas as pd

locations = pd.read_csv('../data/datacenter_locations.csv')
print(f"Loaded {len(locations)} data centre locations")
locations.head()

Loaded 33 data centre locations


,name,company,lat,lon,capacity_mw,wri_stress_score
0,The Dalles,Google,45.5946,-121.1787,200,1.5
1,Council Bluffs,Google,41.2619,-95.8608,350,2.0
2,Lenoir,Google,35.9140,-81.5390,150,1.2
3,Mayes County,Google,36.3000,-95.2000,250,2.8
4,Hamina,Google,60.5693,27.1878,120,0.8


In [2]:
print("=== Water Stress Distribution ===")
high_stress = (locations['wri_stress_score'] >= 3).sum()
total = len(locations)
print(f"High stress (>=3): {high_stress}/{total} ({high_stress/total*100:.0f}%)")
print(f"\nBy company:")
print(locations.groupby('company')['wri_stress_score'].agg(['mean', 'max']).round(2))

=== Water Stress Distribution ===
High stress (>=3): 13/33 (39%)

By company:
           mean  max
company             
AWS        2.64  5.0
Google     2.19  4.2
Meta       2.50  4.5
Microsoft  2.94  4.8


In [3]:
m = folium.Map(location=[25, 0], zoom_start=2,
               tiles='CartoDB dark_matter',
               width='100%', height='100%')

company_icons = {
    'Google': '🔵',
    'Microsoft': '🟢',
    'Meta': '🔷',
    'AWS': '🟠'
}

for _, row in locations.iterrows():
    score = row['wri_stress_score']
    if score >= 4:
        colour = 'darkred'
        stress_label = 'Extremely High'
    elif score >= 3:
        colour = 'red'
        stress_label = 'High'
    elif score >= 2:
        colour = 'orange'
        stress_label = 'Medium-High'
    else:
        colour = 'green'
        stress_label = 'Low'

    radius = max(5, row['capacity_mw'] / 50)

    popup_html = f"""
    <div style="font-family: Arial; width: 200px;">
        <h4 style="margin: 0 0 5px 0;">{row['name']}</h4>
        <b>Company:</b> {row['company']}<br>
        <b>Capacity:</b> {row['capacity_mw']} MW<br>
        <b>Water Stress:</b> {score}/5 ({stress_label})<br>
        <b>Coords:</b> {row['lat']:.2f}, {row['lon']:.2f}
    </div>
    """

    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=radius,
        color=colour,
        fill=True,
        fill_color=colour,
        fill_opacity=0.7,
        weight=2,
        popup=folium.Popup(popup_html, max_width=250),
        tooltip=f"{row['company']} — {row['name']} ({row['capacity_mw']} MW)"
    ).add_to(m)

legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
            background-color: rgba(0,0,0,0.8); padding: 15px; border-radius: 8px;
            font-family: Arial; color: white; font-size: 13px;">
    <b>Water Stress Level (WRI)</b><br>
    <span style="color: green;">●</span> Low (0-2)<br>
    <span style="color: orange;">●</span> Medium-High (2-3)<br>
    <span style="color: red;">●</span> High (3-4)<br>
    <span style="color: darkred;">●</span> Extremely High (4-5)<br>
    <br><b>Dot Size</b> = Facility Capacity (MW)<br>
    <br><i>42% of AI data centres are in<br>high water-stress zones</i>
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

title_html = """
<div style="position: fixed; top: 10px; left: 50%; transform: translateX(-50%); z-index: 1000;
            background-color: rgba(0,0,0,0.85); padding: 12px 25px; border-radius: 8px;
            font-family: Arial; color: white; font-size: 18px; font-weight: bold;">
    AI Data Centres & Water Stress — Global Map
</div>
"""
m.get_root().html.add_child(folium.Element(title_html))

m

In [4]:
m.save('../outputs/datacenter_water_stress_map.html')
print("Map saved: outputs/datacenter_water_stress_map.html")

Map saved: outputs/datacenter_water_stress_map.html
